# Notebook 01 — Load Workshop Data

This notebook creates the manufacturing dataset that all subsequent notebooks use. It writes 7 tables into **your** Unity Catalog schema as Parquet files in a Volume.

**Tables created:**

| Table | Rows | What it contains |
|-------|------|------------------|
| `plants` | 8 | Manufacturing facilities (name, city, state, capacity) |
| `production_lines` | 32 | 4 lines per plant (product type, status, install date) |
| `operators` | 500 | Workers across morning/afternoon/night shifts |
| `production_events` | 50,000 | unit_produced, defect_detected, scrap, rework, downtime events |
| `quality_metrics_daily` | ~2,196 | OEE score, first pass yield, downtime minutes per line per day |
| `safety_incidents` | 40 | Critical/Major/Minor incidents with root cause and corrective action |
| `equipment_feedback` | 100 | Operator feedback on equipment condition |

All data covers calendar year **2024** across 8 plants in Michigan, Ohio, Tennessee, and Texas.

**Runtime:** ~1 minute.

In [ ]:
%run ./00_workshop_config

## Create Volume

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{PREBUILD_VOLUME}")
print(f"Volume ready: {PREBUILD_PATH}")

## Static Data Pools

All names, descriptions, and comments are hardcoded string literals — no Faker dependency.

In [ ]:
import hashlib
from datetime import date, datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType, BooleanType, DateType, TimestampType
)

# ── Name pools (50 first + 50 last = 2,500 unique combos) ──
FIRST_NAMES = [
    "James", "Maria", "Robert", "Linda", "David", "Sarah", "Carlos", "Jennifer", "Michael", "Patricia",
    "William", "Elizabeth", "Richard", "Barbara", "Joseph", "Susan", "Thomas", "Jessica", "Daniel", "Karen",
    "Matthew", "Nancy", "Anthony", "Lisa", "Mark", "Betty", "Donald", "Margaret", "Steven", "Sandra",
    "Paul", "Ashley", "Andrew", "Dorothy", "Joshua", "Kimberly", "Kenneth", "Emily", "Kevin", "Donna",
    "Brian", "Michelle", "George", "Carol", "Timothy", "Amanda", "Ronald", "Melissa", "Edward", "Deborah",
]
LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez",
    "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin",
    "Lee", "Perez", "Thompson", "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson",
    "Walker", "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores",
    "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts",
]

SHIFTS = ["Morning", "Afternoon", "Night"]
CERTIFICATIONS = ["Welding", "Robotics", "Paint", "Quality", "Safety", "EV Systems", "Stamping"]
PRODUCT_TYPES = ["Sedan", "Truck", "SUV", "EV Battery Pack", "Crossover"]
LINE_TYPES = ["Assembly", "Paint", "Stamping", "EV Battery", "Powertrain"]

EVENT_TYPES = [
    "unit_produced", "unit_produced", "unit_produced", "unit_produced",
    "defect_detected", "inspection_passed", "inspection_passed",
    "downtime_start", "scrap", "rework_completed",
]  # 10 entries: 40% produced, 10% defect, 20% inspection, 10% downtime, 10% scrap, 10% rework

DEFECT_CODES = [
    "DEF-WELD-001", "DEF-WELD-002", "DEF-WELD-003",
    "DEF-PAINT-001", "DEF-PAINT-002", "DEF-PAINT-003",
    "DEF-FIT-001", "DEF-FIT-002",
    "DEF-ELEC-001", "DEF-ELEC-002",
    "DEF-STMP-001", "DEF-STMP-002",
]

SEVERITY_LEVELS = ["Low", "Medium", "High", "Critical"]
INCIDENT_CATEGORIES = ["Ergonomic", "Chemical Exposure", "Equipment Malfunction", "Slip/Trip/Fall", "Electrical", "Fire Hazard"]
RESOLUTIONS = ["Resolved", "Under Investigation", "Pending Action", "Closed"]

INCIDENT_DESCRIPTIONS = [
    "Worker reported shoulder strain during repetitive welding operation on assembly line.",
    "Minor chemical splash detected near paint booth ventilation system during shift change.",
    "Robotic arm exceeded safety boundary during calibration sequence and triggered emergency stop.",
    "Operator slipped on oil residue near stamping press area during morning inspection rounds.",
    "Electrical arc detected at junction box during routine maintenance of conveyor motor system.",
    "Small fire extinguished near welding station due to accumulated metal shavings and debris.",
    "Forklift operator reported near-miss with pedestrian at warehouse loading dock intersection area.",
    "Excessive noise levels measured at stamping press exceeding recommended occupational exposure limits.",
    "Coolant leak from CNC machine created slippery surface in machining area near walkway.",
    "Worker experienced dizziness from paint fumes due to temporary ventilation system malfunction.",
    "Hydraulic line failure on press caused unexpected movement during automated stamping cycle.",
    "Eye irritation reported by three operators near chemical storage area after container leak.",
    "Conveyor belt misalignment caused product jam and required emergency shutdown for clearance.",
    "Burn injury from hot metal contact during manual deburring operation on stamped components.",
    "Safety guard found removed from grinding station during routine safety inspection walk.",
    "Compressed air hose failure caused whipping hazard near assembly station during operation.",
    "Worker tripped over misplaced power cable routing across main aisle in assembly area.",
    "Repetitive strain injury reported by quality inspector from prolonged visual inspection tasks.",
    "Battery cell thermal event detected during EV pack assembly requiring immediate evacuation protocol.",
    "Lockout tagout procedure violation found during maintenance of robotic welding cell equipment.",
    "Overhead crane load indicator malfunction during heavy component transfer in assembly bay.",
    "Dust accumulation in electrical panel triggered fire suppression system in stamping department.",
    "Operator caught loose clothing near rotating equipment requiring immediate medical attention.",
    "Paint overspray contamination detected in adjacent clean room assembly area during audit.",
    "Automated guided vehicle collision with stationary equipment in warehouse navigation zone.",
    "Gas leak detected from propane forklift in enclosed loading area requiring ventilation action.",
    "Worker reported back pain from improper ergonomic setup at manual assembly workstation.",
    "Weld spatter ignited nearby cardboard packaging material requiring quick fire suppression response.",
    "Falling tool from overhead platform struck safety railing near worker walking path.",
    "Chemical compatibility issue found during waste storage audit in hazardous materials area.",
    "Robot teaching pendant malfunction caused unexpected arm movement during programming session.",
    "Steam pipe insulation failure released hot vapor near operator walkway in paint department.",
    "Inadequate lighting reported in maintenance corridor increasing trip and fall risk assessment.",
    "Vibration damage to sensitive measurement equipment from nearby stamping press operations.",
    "Operator experienced hearing threshold shift despite wearing standard hearing protection equipment.",
    "Improperly secured compressed gas cylinder found during monthly safety compliance inspection.",
    "Automated door sensor failure trapped worker between production zones during shift transition.",
    "Welding curtain deterioration allowed UV radiation exposure to adjacent workstation personnel.",
    "Floor drain blockage caused chemical runoff pooling in production area near emergency exit.",
    "Battery electrolyte spill during EV module handling required specialized hazmat cleanup response.",
]

ROOT_CAUSES = [
    "Inadequate maintenance schedule for equipment.", "Operator training gap identified.",
    "Worn safety guard not replaced on time.", "Ventilation system below capacity.",
    "Improper chemical storage procedures.", "Equipment calibration overdue.",
    "Insufficient lighting in work area.", "PPE compliance not enforced.",
    "Emergency procedure not followed correctly.", "Design flaw in workstation layout.",
]

CORRECTIVE_ACTIONS = [
    "Scheduled immediate maintenance and updated PM calendar.", "Conducted retraining session for affected operators.",
    "Replaced guard and added inspection to daily checklist.", "Upgraded ventilation and added monitoring sensors.",
    "Revised chemical handling SOP and retrained staff.", "Recalibrated equipment and set monthly reminders.",
    "Installed additional LED lighting in affected areas.", "Implemented daily PPE compliance audits.",
    "Updated emergency procedures and ran drill exercises.", "Redesigned workstation with ergonomic assessment.",
]

EQUIPMENT_NAMES = [
    "Fanuc Robotic Welder", "KUKA Paint Robot", "Schuler Press", "ABB Assembly Arm",
    "Siemens Conveyor", "Tesla Battery Stacker", "Atlas Copco Torque Gun", "Keyence Vision System",
]

FEEDBACK_COMMENTS = [
    "Runs smoothly, no issues this week.",
    "Calibration drifting — needs maintenance soon.",
    "Excellent precision on welds today.",
    "Intermittent faults during second shift.",
    "Paint coverage has been inconsistent lately.",
    "Battery stacker alignment is perfect after recalibration.",
    "Torque gun readings are within spec.",
    "Conveyor belt slipping under heavy load.",
    "Vision system catching defects we used to miss.",
    "Needs replacement — frequent breakdowns.",
    "Outstanding performance after firmware update.",
    "Hydraulic leak detected — reported to maintenance.",
]

_VIN_CHARS = "ABCDEFGHJKLMNPRSTUVWXYZ0123456789"

def _deterministic_vin(event_id: str) -> str:
    """Generate a deterministic 17-char VIN from event_id hash."""
    h = hashlib.sha256(event_id.encode()).hexdigest()
    return "".join(_VIN_CHARS[int(h[i:i+2], 16) % len(_VIN_CHARS)] for i in range(0, 34, 2))

print("Static data pools loaded.")

## Generate Plants (8 rows)

In [ ]:
PLANTS = [
    {"plant_id": "P001", "plant_name": "Midwest Assembly Plant",      "city": "Detroit",     "state": "Michigan",    "country": "USA", "annual_revenue": 2400000000.0, "employee_count": 4500},
    {"plant_id": "P002", "plant_name": "Great Lakes Stamping",        "city": "Flint",       "state": "Michigan",    "country": "USA", "annual_revenue": 1800000000.0, "employee_count": 3200},
    {"plant_id": "P003", "plant_name": "Southern Paint & Body",       "city": "Spring Hill", "state": "Tennessee",   "country": "USA", "annual_revenue": 2100000000.0, "employee_count": 3800},
    {"plant_id": "P004", "plant_name": "EV Battery Innovation Center","city": "Lordstown",   "state": "Ohio",        "country": "USA", "annual_revenue": 950000000.0,  "employee_count": 1200},
    {"plant_id": "P005", "plant_name": "Heartland Powertrain",        "city": "Tonawanda",   "state": "New York",    "country": "USA", "annual_revenue": 1600000000.0, "employee_count": 2900},
    {"plant_id": "P006", "plant_name": "Delta Assembly",              "city": "Lansing",     "state": "Michigan",    "country": "USA", "annual_revenue": 2200000000.0, "employee_count": 4100},
    {"plant_id": "P007", "plant_name": "Lone Star Truck Assembly",    "city": "Arlington",   "state": "Texas",       "country": "USA", "annual_revenue": 3100000000.0, "employee_count": 5200},
    {"plant_id": "P008", "plant_name": "Pacific Coast Paint Facility","city": "Fremont",     "state": "California",  "country": "USA", "annual_revenue": 1400000000.0, "employee_count": 2200},
]
print(f"Plants: {len(PLANTS)} rows")

## Generate Production Lines (32 rows)

4 lines per plant, deterministic type/product assignment.

In [ ]:
lines = []
line_counter = 1
# Fixed: 4 lines per plant, types cycle through LINE_TYPES
for pi, plant in enumerate(PLANTS):
    plant_short = plant["plant_name"].split()[0]
    for li in range(4):
        lt = LINE_TYPES[li % len(LINE_TYPES)]
        pt = PRODUCT_TYPES[li % len(PRODUCT_TYPES)]
        # Deterministic capacity: 300 + (line_counter * 17) % 500
        capacity = 300 + (line_counter * 17) % 500
        cost = round(80.0 + (line_counter * 13.7) % 400, 2)
        # Deterministic start_date: spread across 2019-2023
        start_days = 365 * 2 + (line_counter * 47) % (365 * 3)  # 2-5 years ago from 2024
        start_date = date(2024, 1, 1) - timedelta(days=start_days)
        status = "Maintenance" if line_counter % 8 == 0 else "Active"
        lines.append({
            "line_id": f"L{line_counter:03d}",
            "line_name": f"{plant_short} {lt} Line {li + 1}",
            "description": f"{lt} line producing {pt} components",
            "product_type": pt,
            "plant_id": plant["plant_id"],
            "daily_capacity": capacity,
            "cost_per_unit": cost,
            "start_date": start_date,
            "end_date": None,
            "status": status,
        })
        line_counter += 1

print(f"Production lines: {len(lines)} rows")

## Generate Operators (500 rows)

Names from static pool, shift/cert/plant by modular arithmetic.

In [ ]:
operators = []
for i in range(1, 501):
    first = FIRST_NAMES[i % len(FIRST_NAMES)]
    last = LAST_NAMES[(i * 7) % len(LAST_NAMES)]  # offset to avoid same-index combos
    shift = SHIFTS[i % len(SHIFTS)]
    cert = CERTIFICATIONS[i % len(CERTIFICATIONS)]
    plant_id = PLANTS[i % len(PLANTS)]["plant_id"]
    # Deterministic hire_date: spread 2009-2023
    hire_days = 365 + (i * 31) % (365 * 14)  # 1-15 years ago
    hire_date = date(2024, 1, 1) - timedelta(days=hire_days)
    operators.append({
        "operator_id": f"OP{i:04d}",
        "first_name": first,
        "last_name": last,
        "shift": shift,
        "certification": cert,
        "hire_date": hire_date,
        "plant_id": plant_id,
    })

print(f"Operators: {len(operators)} rows")

## Generate Production Events (50,000 rows)

Event type, line, operator, date, VIN — all deterministic via modular arithmetic and hash.

In [ ]:
start_dt = date(2024, 1, 1)
delta_days = 365  # 2024 is a leap year = 366 days, use 365 for clean modulo

events = []
for i in range(1, 50001):
    event_id = f"EVT{i:06d}"
    et = EVENT_TYPES[i % len(EVENT_TYPES)]
    day_offset = i % 366
    d = start_dt + timedelta(days=day_offset)
    hour = (i * 3) % 24
    minute = (i * 7) % 60
    ts = datetime(d.year, d.month, d.day, hour, minute)
    line = lines[i % len(lines)]
    op = operators[i % len(operators)]
    vin = _deterministic_vin(event_id)
    defect_code = DEFECT_CODES[i % len(DEFECT_CODES)] if et == "defect_detected" else None
    events.append({
        "event_id": event_id,
        "event_type": et,
        "event_date": d,
        "event_timestamp": ts,
        "production_line_id": line["line_id"],
        "operator_id": op["operator_id"],
        "part_number": f"PN-{10000 + (i * 13) % 90000}",
        "unit_serial_vin": vin,
        "defect_code": defect_code,
    })

print(f"Production events: {len(events)} rows")

## Generate Quality Metrics Daily (~2,196 rows)

OEE and FPY computed from deterministic hash of (line_id + date).

In [ ]:
quality_metrics = []
for day_offset in range(366):  # all of 2024
    d = start_dt + timedelta(days=day_offset)
    # 6 lines per day, deterministic selection
    for li in range(6):
        line_idx = (day_offset * 7 + li) % len(lines)
        line = lines[line_idx]
        # Deterministic values from hash
        key = f"{line['line_id']}_{d.isoformat()}"
        h = int(hashlib.md5(key.encode()).hexdigest()[:8], 16)
        units = 100 + h % 400  # 100-499
        defect_pct = 0.005 + (h % 75) / 1000.0  # 0.5% - 8%
        defects = int(units * defect_pct)
        oee = round(0.60 + (h % 38) / 100.0, 4)  # 0.60-0.97
        fpy = round((units - defects) / units, 4)
        downtime = round((h % 120) + (h % 37) / 10.0, 1)  # 0-120 min
        quality_metrics.append({
            "date": d,
            "plant_id": line["plant_id"],
            "production_line_id": line["line_id"],
            "units_produced": units,
            "units_passed_inspection": units - defects,
            "defects_found": defects,
            "downtime_minutes": downtime,
            "scrap_count": max(0, defects // 2),
            "rework_count": max(0, defects),
            "first_pass_yield": fpy,
            "oee_score": oee,
        })

print(f"Quality metrics: {len(quality_metrics)} rows")

## Generate Safety Incidents (40 rows)

In [ ]:
safety = []
for i in range(1, 41):
    line = lines[i % len(lines)]
    incident_day = 10 + (i * 9) % 350  # spread across 2024
    safety.append({
        "incident_id": f"SI{i:04d}",
        "description": INCIDENT_DESCRIPTIONS[i - 1],  # exactly 40 static descriptions
        "severity": SEVERITY_LEVELS[i % len(SEVERITY_LEVELS)],
        "production_line_id": line["line_id"],
        "incident_date": start_dt + timedelta(days=incident_day),
        "resolution_status": RESOLUTIONS[i % len(RESOLUTIONS)],
        "category": INCIDENT_CATEGORIES[i % len(INCIDENT_CATEGORIES)],
        "root_cause": ROOT_CAUSES[i % len(ROOT_CAUSES)],
        "corrective_action": CORRECTIVE_ACTIONS[i % len(CORRECTIVE_ACTIONS)],
    })

print(f"Safety incidents: {len(safety)} rows")

## Generate Equipment Feedback (100 rows)

In [ ]:
feedback = []
for i in range(1, 101):
    line = lines[i % len(lines)]
    op = operators[(i * 3) % len(operators)]
    fb_day = 5 + (i * 3) % 360
    feedback.append({
        "feedback_id": f"FB{i:04d}",
        "equipment_name": EQUIPMENT_NAMES[i % len(EQUIPMENT_NAMES)],
        "comment": FEEDBACK_COMMENTS[i % len(FEEDBACK_COMMENTS)],
        "feedback_date": start_dt + timedelta(days=fb_day),
        "production_line_id": line["line_id"],
        "operator_id": op["operator_id"],
    })

print(f"Equipment feedback: {len(feedback)} rows")

## Write all 7 tables as Parquet to Volume

Each table is written as a Parquet directory under `{PREBUILD_PATH}/{table_name}/`.

In [ ]:
import pandas as pd
import numpy as np

table_data = {
    "plants": PLANTS,
    "production_lines": lines,
    "operators": operators,
    "production_events": events,
    "quality_metrics_daily": quality_metrics,
    "safety_incidents": safety,
    "equipment_feedback": feedback,
}

for table_name, data in table_data.items():
    path = f"{PREBUILD_PATH}/{table_name}"
    pdf = pd.DataFrame(data)
    # Force all-null object columns to a typed nullable dtype so Spark
    # doesn't infer NullType (which Parquet cannot write).
    for col in pdf.columns:
        if pdf[col].isna().all():
            pdf[col] = pd.array([pd.NaT] * len(pdf), dtype="datetime64[ns]")
    df = spark.createDataFrame(pdf)
    df.write.mode("overwrite").parquet(path)
    count = spark.read.parquet(path).count()
    print(f"  {table_name}: {count} rows -> {path}")

print(f"\nAll 7 tables written to {PREBUILD_PATH}/")

## Verification Checksums

Print checksums so facilitators can verify identical data across environments.

In [ ]:
print("=" * 60)
print("PREBUILD VERIFICATION CHECKSUMS")
print("=" * 60)
for table_name in table_data:
    path = f"{PREBUILD_PATH}/{table_name}"
    df = spark.read.parquet(path)
    count = df.count()
    cols = len(df.columns)
    print(f"  {table_name:30s} | rows={count:>6d} | cols={cols}")
print("=" * 60)
print("\nIf these numbers match across environments, data is identical.")
print("Share this output with participants so they can verify their load.")